# Week 3: TF-IDF, Word2Vec & MinHash LSH Feature Engineering

**Project**: Explainable Fake Review Detection for Bangla
**Course**: CSE 4262 Data Analytics Lab

This notebook implements Week 3:
1. TF-IDF vectors (sparse) for linear models
2. Word2Vec dense embeddings for tree models
3. MinHash LSH for near-duplicate detection
4. Combine all features into model-ready format
5. Save fitted pipelines and feature matrices

**Key Design Decisions**:
- TF-IDF: fitted on gold train+val only (no leakage)
- Word2Vec: fitted on gold train+val only  
- MinHash: binary shingles (k=3) for Jaccard similarity
- Separate feature sets for linear vs tree models

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
!pip install -q pyspark==3.5.0 2>/dev/null

import json
from pathlib import Path
from datetime import datetime

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import (
    CountVectorizer, IDF, Word2Vec, MinHashLSH,
    VectorAssembler, StandardScaler, Normalizer
)
from pyspark.ml import Pipeline
from pyspark.ml.linalg import VectorUDT, Vectors

print("Dependencies loaded")

In [ ]:
# ============================================================
# SPARK SESSION
# ============================================================
spark = (SparkSession.builder
    .appName("BanglaFakeReview_W3")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

In [ ]:
# ============================================================
# PATHS & LOAD DATA
# ============================================================
BASE_DIR = Path("/kaggle/working/bangla_fake_review")
PROCESSED_DIR = BASE_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

gold = spark.read.parquet(str(PROCESSED_DIR / "bfrd_gold_features.parquet"))
bangrev = spark.read.parquet(str(PROCESSED_DIR / "bangrev_features.parquet"))
test = spark.read.parquet(str(PROCESSED_DIR / "bfrd_test_features.parquet"))

print(f"Gold: {gold.count()}")
print(f"BanglishRev: {bangrev.count()}")
print(f"Test: {test.count()}")

In [ ]:
# ============================================================
# 1. SPLIT GOLD INTO TRAIN/VAL (already has source_split)
# ============================================================
train = gold.filter(F.col("source_split") == "train")
val = gold.filter(F.col("source_split") == "validation")

print(f"Train: {train.count()}")
print(f"Val: {val.count()}")

In [ ]:
# ============================================================
# 2. TF-IDF PIPELINE (fitted on train only)
# ============================================================
# CountVectorizer on tokens
cv = CountVectorizer(
    inputCol="tokens",
    outputCol="tf",
    vocabSize=50000,
    minDF=3.0,
    minTF=1.0,
    binary=False
)

# IDF
idf = IDF(
    inputCol="tf",
    outputCol="tfidf",
    minDocFreq=3
)

# Fit on TRAIN only
print("Fitting CountVectorizer on train...")
cv_model = cv.fit(train)
print(f"Vocabulary size: {len(cv_model.vocabulary)}")

train_tf = cv_model.transform(train)
val_tf = cv_model.transform(val)
test_tf = cv_model.transform(test)
bangrev_tf = cv_model.transform(bangrev)

print("Fitting IDF on train...")
idf_model = idf.fit(train_tf)

train_tfidf = idf_model.transform(train_tf)
val_tfidf = idf_model.transform(val_tf)
test_tfidf = idf_model.transform(test_tf)
bangrev_tfidf = idf_model.transform(bangrev_tf)

# Save models
cv_model.write().overwrite().save(str(MODELS_DIR / "countvectorizer"))
idf_model.write().overwrite().save(str(MODELS_DIR / "idf"))

print("TF-IDF complete")

In [ ]:
# ============================================================
# 3. WORD2VEC PIPELINE (fitted on train only)
# ============================================================
w2v = Word2Vec(
    inputCol="tokens",
    outputCol="w2v_vec",
    vectorSize=100,
    minCount=3,
    numPartitions=4,
    stepSize=0.025,
    maxIter=10,
    seed=42
)

print("Fitting Word2Vec on train...")
w2v_model = w2v.fit(train)

train_w2v = w2v_model.transform(train)
val_w2v = w2v_model.transform(val)
test_w2v = w2v_model.transform(test)
bangrev_w2v = w2v_model.transform(bangrev)

w2v_model.write().overwrite().save(str(MODELS_DIR / "word2vec"))

print(f"Word2Vec vocab: {len(w2v_model.getVectors())}")
print("Word2Vec complete")

In [ ]:
# ============================================================
# 4. MINHASH LSH FOR NEAR-DUPLICATES
# ============================================================
# Convert tokens to binary shingles (k=3 character n-grams)
# For MinHash, we need a binary vector representation of shingles

from pyspark.ml.feature import NGram, HashingTF

# Create character 3-grams from clean_review
ngram = NGram(n=3, inputCol="clean_review", outputCol="char_3grams")

# HashingTF to convert to fixed-size binary sparse vectors
hashing_tf = HashingTF(
    inputCol="char_3grams",
    outputCol="shingle_vec",
    numFeatures=8192,  # Power of 2 for MinHash
    binary=True  # Binary for MinHash
)

# Apply to all datasets
for name, df in [("train", train), ("val", val), ("test", test), ("bangrev", bangrev)]:
    df_ngram = ngram.transform(df)
    df_shingle = hashing_tf.transform(df_ngram)
    # Cache for MinHash
    df_shingle.cache()
    globals()[f"{name}_shingle"] = df_shingle
    print(f"{name} shingles: {df_shingle.count()}")

In [ ]:
# Fit MinHashLSH on train shingles
mh = MinHashLSH(
    inputCol="shingle_vec",
    outputCol="minhash_sig",
    numHashTables=5,  # More tables = better recall
    seed=42
)

print("Fitting MinHashLSH...")
mh_model = mh.fit(train_shingle)

# Transform all
train_mh = mh_model.transform(train_shingle)
val_mh = mh_model.transform(val_shingle)
test_mh = mh_model.transform(test_shingle)
bangrev_mh = mh_model.transform(bangrev_shingle)

mh_model.write().overwrite().save(str(MODELS_DIR / "minhash_lsh"))

print("MinHash LSH fitted")

In [ ]:
# ============================================================
# 5. NEAR-DUPLICATE NEIGHBOR SEARCH
# ============================================================
# Find approximate Jaccard neighbors (threshold ~0.7)
# This is expensive - we'll do it on train vs train for graph features

print("Finding near-duplicate neighbors (Jaccard > 0.7)...")

# Self-join on train to find duplicate pairs
neighbors = mh_model.approxSimilarityJoin(
    train_mh, train_mh,
    threshold=0.3,  # Distance threshold (1 - Jaccard), so 0.3 = Jaccard > 0.7
    distCol="jaccard_dist"
)

# Filter self-matches and keep only datasetA.review_id < datasetB.review_id
dup_pairs = neighbors.filter(
    (F.col("datasetA.review_id") < F.col("datasetB.review_id"))
).select(
    F.col("datasetA.review_id").alias("review_id_a"),
    F.col("datasetB.review_id").alias("review_id_b"),
    (1 - F.col("jaccard_dist")).alias("jaccard_sim")
)

dup_pairs_count = dup_pairs.count()
print(f"Near-duplicate pairs (Jaccard > 0.7): {dup_pairs_count}")

# Compute graph features: degree and cluster size
from pyspark.sql import Window

# Degree = number of neighbors
degree_a = dup_pairs.groupBy("review_id_a").count().withColumnRenamed("count", "dup_degree")
degree_b = dup_pairs.groupBy("review_id_b").count().withColumnRenamed("count", "dup_degree")
degrees = degree_a.unionByName(degree_b).groupBy("review_id_a").sum("dup_degree")
    .withColumnRenamed("review_id_a", "review_id")
    .withColumnRenamed("sum(dup_degree)", "dup_degree")

# Connected components for cluster size (approximate via iterative join)
# Simple approach: use the max component size from the pair graph
# For now, use degree as proxy

# Join back to train
train_mh = train_mh.join(degrees, "review_id", "left").fillna({ "dup_degree": 0 })
    .withColumn("dup_cluster_size", F.col("dup_degree") + 1)

# For val/test/bangrev: find neighbors in TRAIN only
for name, df_mh in [("val", val_mh), ("test", test_mh), ("bangrev", bangrev_mh)]:
    cross_neighbors = mh_model.approxSimilarityJoin(
        df_mh, train_mh,
        threshold=0.3,
        distCol="jaccard_dist"
    )
    cross_pairs = cross_neighbors.select(
        F.col("datasetA.review_id").alias("review_id"),
        (1 - F.col("jaccard_dist")).alias("jaccard_sim")
    )
    cross_degree = cross_pairs.groupBy("review_id").count()
        .withColumnRenamed("count", "dup_degree")
    df_mh = df_mh.join(cross_degree, "review_id", "left")
        .fillna({ "dup_degree": 0 })
        .withColumn("dup_cluster_size", F.col("dup_degree") + 1)
    globals()[f"{name}_mh"] = df_mh
    print(f"{name} cross neighbors computed")

In [ ]:
# ============================================================
# 6. HANDCRAFTED FEATURE VECTOR ASSEMBLY
# ============================================================
# Numeric handcrafted features for tree models (Word2Vec + engineered)
hc_numeric = [
    "char_length", "token_count", "codemix_ratio", "repeated_char_ratio",
    "url_count", "phone_count", "price_count", "hashtag_count",
    "exclamation_count", "emoji_count", "promo_keyword_hits",
    "bengali_chars", "latin_chars", "digit_chars", "other_chars",
    "exact_dup_cluster_size", "dup_degree", "dup_cluster_size"
]

# Binary flags
hc_binary = ["has_url", "has_phone", "has_price", "has_promo", "has_emoji", "is_exact_duplicate"]

all_hc = hc_numeric + hc_binary

# Assemble for each dataset
assembler_hc = VectorAssembler(inputCols=all_hc, outputCol="hc_vec_raw", handleInvalid="skip")

for name, df in [("train", train_mh), ("val", val_mh), ("test", test_mh), ("bangrev", bangrev_mh)]:
    df_assembled = assembler_hc.transform(df)
    globals()[f"{name}_hc"] = df_assembled
    print(f"{name} handcrafted assembled")

In [ ]:
# Scale handcrafted features (fit on train only)
scaler = StandardScaler(
    inputCol="hc_vec_raw",
    outputCol="hc_vec",
    withStd=True,
    withMean=False  # Sparse-friendly
)

scaler_model = scaler.fit(train_hc)

train_hc = scaler_model.transform(train_hc)
val_hc = scaler_model.transform(val_hc)
test_hc = scaler_model.transform(test_hc)
bangrev_hc = scaler_model.transform(bangrev_hc)

scaler_model.write().overwrite().save(str(MODELS_DIR / "hc_scaler"))

print("Handcrafted features scaled")

In [ ]:
# ============================================================
# 7. FINAL FEATURE MATRICES FOR EACH MODEL TYPE
# ============================================================
# Linear models (LR, Linear SVC, NB): TF-IDF + handcrafted
assembler_linear = VectorAssembler(
    inputCols=["tfidf", "hc_vec"],
    outputCol="features_linear",
    handleInvalid="skip"
)

# Tree models (RF, GBT): Word2Vec + handcrafted
assembler_tree = VectorAssembler(
    inputCols=["w2v_vec", "hc_vec"],
    outputCol="features_tree",
    handleInvalid="skip"
)

# Naive Bayes needs non-negative features: TF-IDF (already non-neg) + abs(handcrafted)
# We'll handle this in modeling

for name, df in [("train", train_hc), ("val", val_hc), ("test", test_hc), ("bangrev", bangrev_hc)]:
    # Need to join with tfidf and w2v
    tfidf_df = globals()[f"{name}_tfidf"].select("review_id", "tfidf")
    w2v_df = globals()[f"{name}_w2v"].select("review_id", "w2v_vec")
    
    df_joined = df.join(tfidf_df, "review_id").join(w2v_df, "review_id")
    
    df_linear = assembler_linear.transform(df_joined)
    df_tree = assembler_tree.transform(df_joined)
    
    globals()[f"{name}_linear"] = df_linear
    globals()[f"{name}_tree"] = df_tree
    print(f"{name} feature matrices ready")

In [ ]:
# ============================================================
# 8. PREPARE NAIVE BAYES FEATURES (non-negative only)
# ============================================================
# For NB: use CountVectorizer output (tf) + non-negative handcrafted
# TF is already non-negative. Handcrafted needs to be non-negative.

from pyspark.ml.feature import MinMaxScaler

nb_scaler = MinMaxScaler(inputCol="hc_vec_raw", outputCol="hc_vec_nb")
nb_scaler_model = nb_scaler.fit(train_hc)

for name, df in [("train", train_hc), ("val", val_hc), ("test", test_hc), ("bangrev", bangrev_hc)]:
    tf_df = globals()[f"{name}_tfidf"].select("review_id", "tf")  # tf, not tfidf
    df_scaled = nb_scaler_model.transform(df)
    df_joined = df_scaled.join(tf_df, "review_id")
    
    assembler_nb = VectorAssembler(
        inputCols=["tf", "hc_vec_nb"],
        outputCol="features_nb",
        handleInvalid="skip"
    )
    df_nb = assembler_nb.transform(df_joined)
    globals()[f"{name}_nb"] = df_nb
    print(f"{name} NB features ready")

In [ ]:
# ============================================================
# 9. SAVE MODEL-READY DATASETS
# ============================================================
cols_to_keep = ["review_id", "label", "source_dataset", "source_split", "label_source",
               "confidence_score", "features_linear", "features_tree", "features_nb"]

for name in ["train", "val", "test", "bangrev"]:
    df = globals()[f"{name}_linear"]
    # Select only needed columns
    keep_cols = [c for c in cols_to_keep if c in df.columns]
    df_final = df.select(*keep_cols)
    df_final.write.mode("overwrite").parquet(str(PROCESSED_DIR / f"{name}_model_ready.parquet"))
    print(f"Saved {name}_model_ready: {df_final.count()} rows")

In [ ]:
# ============================================================
# 10. FEATURE STATISTICS & VOCABULARY INSPECTION
# ============================================================
# TF-IDF vocab top terms by class
vocab = cv_model.vocabulary
print(f"TF-IDF vocab size: {len(vocab)}")
print(f"First 20 terms: {vocab[:20]}")

# Word2Vec vocabulary
w2v_vectors = w2v_model.getVectors()
print(f"Word2Vec vocab size: {len(w2v_vectors)}")
print(f"Sample vectors: {w2v_vectors.head(5) if hasattr(w2v_vectors, 'head') else list(w2v_vectors.items())[:5]}")

# Feature dimensions
train_linear = spark.read.parquet(str(PROCESSED_DIR / "train_model_ready.parquet"))
sample_vec = train_linear.select("features_linear").first()["features_linear"]
print(f"Linear features dim: {sample_vec.size}")
print(f"  Non-zero: {sample_vec.numNonzeros()}")

sample_vec_tree = train_linear.select("features_tree").first()["features_tree"]
print(f"Tree features dim: {sample_vec_tree.size}")

In [ ]:
# ============================================================
# 11. WEEK 3 SUMMARY
# ============================================================
summary = {
    "week": 3,
    "timestamp": datetime.now().isoformat(),
    "tfidf_vocab_size": len(cv_model.vocabulary),
    "tfidf_min_df": 3,
    "word2vec_vector_size": 100,
    "word2vec_vocab_size": len(w2v_model.getVectors()),
    "minhash_num_hash_tables": 5,
    "minhash_threshold": 0.3,
    "minhash_shingle_n": 3,
    "minhash_num_features": 8192,
    "handcrafted_features": all_hc,
    "linear_feature_dim": sample_vec.size,
    "tree_feature_dim": sample_vec_tree.size,
    "models_saved": [
        "models/countvectorizer",
        "models/idf",
        "models/word2vec",
        "models/minhash_lsh",
        "models/hc_scaler",
        "models/nb_scaler"
    ],
    "data_saved": [
        "processed/train_model_ready.parquet",
        "processed/val_model_ready.parquet",
        "processed/test_model_ready.parquet",
        "processed/bangrev_model_ready.parquet"
    ]
}

summary_path = BASE_DIR / "audit" / f"week3_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Week 3 complete. Summary saved to {summary_path}")